
To install, ´conda env create -f environment-HGQ.yml´, or to update env ´conda env update -f environment-HGQ.yml´. Remember to restart kernel.

In [1]:
import os
model_to_test = 'hgq2'
model_revision = '2'
hls4ml_revision = 'VitisUnified_2025'

base_dir = os.path.abspath(model_to_test)
model_dir = os.path.join(base_dir, model_revision)
os.makedirs(model_dir, exist_ok=True)

description = """
# Model Configuration

Aim is to run inference on HW (VitisUnified with custom 2025-script-patch)
Problems running HGQ2-models; Vitis Unified sets io_stream, but HGQ2 requires io_parallel for heteregenous activation. 
This is just to test different models.

- **Model architecture description**: {model_to_test}
- **Model Revision**: {model_revision}
- **HLS4ML Revision**: {hls4ml_revision}
- **Target Device**: KV260 (xck26-sfvc784-2LV-c)
- **Dataset**: HLS4ML LHC Jets
- **Vivado/Vitis**: 2025.2
"""
output_dir = os.path.join(model_dir, f"hls4ml_prj_{hls4ml_revision}")
os.makedirs(output_dir, exist_ok=True)
with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
    f.write(description)

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
from sklearn.metrics import accuracy_score

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

2026-03-23 20:14:58.992066: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']
!vitis --version
!vitis_hls -version
!vivado -version


****** Vitis Development Environment
****** Vitis v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:14
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

/bin/bash: line 1: vitis_hls: command not found
vivado v2025.2 (64-bit)
Tool Version Limit: 2025.11
SW Build 6299465 on Fri Nov 14 12:34:56 MST 2025
IP Build 6300035 on Fri Nov 14 10:48:45 MST 2025
SharedData Build 6298862 on Thu Nov 13 04:50:51 MST 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.


In [5]:
# Use absolute paths for data files
x_train_val_path = os.path.join(base_dir, "x_train_val.npy")
x_test_path = os.path.join(base_dir, "x_test.npy")
y_train_val_path = os.path.join(base_dir, "y_train_val.npy")
y_test_path = os.path.join(base_dir, "y_test.npy")
classes_path = os.path.join(base_dir, "classes.npy")

x_train_val = np.load(x_train_val_path)
x_test = np.load(x_test_path)
y_train_val = np.load(y_train_val_path)
y_test = np.load(y_test_path)

x_test.dtype

dtype('float32')

In [6]:
# Prepare subset of testdata for simulation (running everything takes an unnecessary long time)
simulation_rows = 100
x_test_sim_path = os.path.join(base_dir, "x_test_sim.npy")
y_test_sim_path = os.path.join(base_dir, "y_test_sim.npy")
np.save(x_test_sim_path, x_test[:simulation_rows])
np.save(y_test_sim_path, y_test[:simulation_rows])

In [ ]:
keras_model_path = os.path.join(model_dir, f"model_HGQ.keras")

import hgq.layers
from keras.models import load_model
model = load_model(keras_model_path)

/home/ncgadmin/miniconda3/envs/devenv-hgq/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 31 variables whereas the saved optimizer has 60 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [8]:
# Save the model summary to a text file
with open(os.path.join(model_dir, "summary.txt"), "w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line: f.write(line + "\n"))

# Convert and synthesize with HLS4ML
Configure parameters.
KV260: xck26-sfvc784-2LV-c 

In [9]:
import hls4ml

hls_config = hls4ml.utils.config_from_keras_model(
    model, 
    backend='vitisunified',
    )

hls_config['Model']['ReuseFactor'] = 4
hls_config['Model']['Strategy'] = 'latency'

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitisunified',
    hls_config=hls_config,
    project_name=f'{model_to_test}_{model_revision}_hls4ml_prj_{hls4ml_revision}',
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    # Set input data for model simulation
    input_data_tb= x_test_sim_path,
    output_data_tb=y_test_sim_path,
)
hls_model.compile()
#hls4ml.utils.plot_model(hls_model, show_shapes=True, show_precision=True,to_file=os.path.join(output_dir, "model-plot.png"))

NotImplementedError: Heterogenous quantization for activations is only supported with IOType=io_parallel

Check performance

In [13]:
y_keras = model.predict(x_test)
y_hls = hls_model.predict(np.ascontiguousarray(x_test))

print("Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):")
for x,y in enumerate(y_keras[:5]):
    print(f"{y-y_hls[x]}")

#print(y_keras[:10])
#print(y_hls[:10])
print("Keras  Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_keras, axis=1))))
print("hls4ml Accuracy: {}".format(accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))))



   4/5188 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step

5188/5188 ━━━━━━━━━━━━━━━━━━━━ 1s 229us/step
Difference in inference-calculations between Keras-model and HLS4ML-compiled model (first rows):
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
Keras  Accuracy: 0.755421686746988
hls4ml Accuracy: 0.755421686746988


In [15]:
hls_model.build(
    csim=False,
    #synth=True, 
    #bitfile=True
    ) 


****** vitis-run v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [vitis-run 82-31] Launching vitis_hls: vitis_hls -nolog -run tcl -f /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_2023_latency_reusefactor4/build_prj.tcl -work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_2023_latency_reusefactor4

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2023.2 (64-bit)
  **** SW Build 4023990 on Oct 11 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis_HLS/2023.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 

{'CSynthesisReport': {'TargetClockPeriod': '5.00',
  'EstimatedClockPeriod': '3.345',
  'BestLatency': '11',
  'WorstLatency': '11',
  'IntervalMin': '2',
  'IntervalMax': '2',
  'DSP': '5',
  'FF': '2709',
  'LUT': '12496',
  'BRAM_18K': '0',
  'URAM': '0',
  'AvailableBRAM_18K': '288',
  'AvailableDSP': '1248',
  'AvailableFF': '234240',
  'AvailableLUT': '117120',
  'AvailableURAM': '64'}}

In [ ]:
hls4ml.report.read_vivado_report(os.path.join(output_dir))

Found 1 solution(s) in /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_2023_latency_reusefactor4/hgq2_1_hls4ml_prj_Vitis_2023_latency_reusefactor4_prj.
Reports for solution "solution1":

C simulation report not found.
SYNTHESIS REPORT:
== Vitis HLS Report for 'hgq2_1_hls4ml_prj_Vitis_2023_latency_reusefactor4'
* Date:           Wed Mar 18 19:36:32 2026

* Version:        2023.2 (Build 4023990 on Oct 11 2023)
* Project:        hgq2_1_hls4ml_prj_Vitis_2023_latency_reusefactor4_prj
* Solution:       solution1 (Vivado IP Flow Target)
* Product family: zynquplus
* Target device:  xck26-sfvc784-2LV-c


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+---------+----------+------------+
    |  Clock |  Target | Estimated| Uncertainty|
    +--------+---------+----------+------------+
    |ap_clk  |  5.00 ns|  3.345 ns|     1.35 ns|
    +--------+---------+----------+------------+

+ Latency: 
    * Summary: 
    +---------+---------+----------

# Simulation
[vitis unified tutorial](https://github.com/Tanawin1701d/vitis_unified_backend_tutorial/blob/master/03_co_simulation.ipynb)

Ubuntu 24 not supported compiling files for co-sim (glibc missmatch). Using 22-docker for running the cosim.

```bash
sudo docker run  -it -d -p 8888:8888 --name pyct -v /home/ncgadmin/Bachelor:/workspace   -v /tools/Xilinx:/tools/Xilinx   -w /workspace   ubuntu:22.04 bash

sudo docker exec -ti pyct bash

apt update

# Vitis installasjongreier
apt-get install -y locales
sed -i 's/^# *en_US.UTF-8 UTF-8/en_US.UTF-8 UTF-8/' /etc/locale.gen
locale-gen
update-locale LANG=en_US.UTF-8 LC_ALL=en_US.UTF-8

bash /tools/Xilinx/Vitis/2023.2/scripts/installLibs.sh

# Python greier
apt install python3-pip python3 git

curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

bash Miniconda3-latest-Linux-x86_64.sh


curl -O https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh

cd HLS4ML_testbench_KV260/development/

source /tools/Xilinx/Vitis_HLS/2023.2/settings64.sh
source /tools/Xilinx/Vivado/2023.2/settings64.sh
source /tools/Xilinx/Model_Composer/2023.2/settings64.sh
source /tools/Xilinx/Vitis/2023.2/settings64.sh

conda activate devenv-vu
jupyter notebook --allow-root --ip=0.0.0.0


In [16]:
# Build and do co-simulation
hls_model.build(
    synth=True, # Only needs to run first time
    cosim=True,
    ) 

# Problem 1 19.03.2026
# Version missmatch mellom system glibc og Vitis 2023.2 (binutils)
# Kjøre i Ubuntu 22 docker, se over


****** vitis-run v2023.2 (64-bit)
  **** SW Build 4026344 on 2023-10-11-15:42:10
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [vitis-run 82-31] Launching vitis_hls: vitis_hls -nolog -run tcl -f /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_2023_latency_reusefactor4/build_prj.tcl -work_dir /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/hgq2/1/hls4ml_prj_Vitis_2023_latency_reusefactor4

****** Vitis HLS - High-Level Synthesis from C, C++ and OpenCL v2023.2 (64-bit)
  **** SW Build 4023990 on Oct 11 2023
  **** IP Build 4028589 on Sat Oct 14 00:45:43 MDT 2023
  **** SharedData Build 4025554 on Tue Oct 10 17:18:54 MDT 2023
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2023 Advanced Micro Devices, Inc. All Rights Reserved.

source /tools/Xilinx/Vitis_HLS/2023.2/scripts/vitis_hls/hls.tcl -notrace
INFO: [HLS 

Exception: Build failed for hgq2_1_hls4ml_prj_Vitis_2023_latency_reusefactor4. See logs for details.

In [ ]:
y_baseline = hls_model.predict(np.ascontiguousarray(x_test[:simulation_rows]))
#y_baseline = model.predict(np.ascontiguousarray(x_test[:simulation_rows]))
y_simulation = np.loadtxt(os.path.join(output_dir, "tb_data/rtl_cosim_results.log"))

In [ ]:
print(f"y_baseline shape: {y_baseline.shape} and y_simulation: {y_simulation.shape}")
#print(y_simulation)

y_baseline shape: (100, 5) and y_simulation: (100, 5)


In [ ]:
assert np.allclose(y_baseline, y_simulation, rtol=0.0, atol=1e-4), (
    "The results from bridge and cosim are NOT equal!"
)
print("\n✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).")


✅ RTL co-simulation comparison passed, absolute difference is less than 1e-4 (atol=1e-4).


In [ ]:
from sklearn.metrics import accuracy_score
print("Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):")
abs_diff = y_baseline[:3] - y_simulation[:3]
print(np.round(abs_diff, 8))

mse = np.mean(np.square(y_baseline - y_simulation))
print(f"MSE: {mse}")


print("Baseline  Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_baseline, axis=1))))
print("Simulation Accuracy: {}".format(accuracy_score(np.argmax(y_test[:simulation_rows], axis=1), np.argmax(y_simulation, axis=1))))

Difference in inference-calculations between HLS4ML bridge and simulated inference (first rows):
[[ 0.0e+00  3.7e-07  3.0e-08  3.0e-08  1.9e-07]
 [-5.0e-08  1.9e-07 -0.0e+00 -2.5e-07  3.7e-07]
 [ 1.2e-07 -4.0e-08 -3.1e-07  0.0e+00  0.0e+00]]
MSE: 2.8122890998877146e-14
Baseline  Accuracy: 0.75
Simulation Accuracy: 0.75
